# Chapter 11 &mdash; Mixed Linearity Need Not Be Regular

**Concept 15 of the Chapter 11 decomposition:** *Mixed Linearity Need Not Be Regular*

`S -> ""`, `S -> (A`, `A -> S)` is a <i>linear</i> grammar for the non-regular MiniDyck.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter11/Concept-Mixed-Linearity/Concept-Mixed-Linearity.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Right-linear grammars give regular languages (Concept 13); so do left-linear ones. It
is tempting to conclude that **linear** grammars &mdash; at most one nonterminal per
right-hand side, anywhere &mdash; also give regular languages.

**They do not.** Consider

```
S -> ''      S -> (A      A -> S)
```

Every right-hand side has at most one nonterminal, so the grammar is linear. But it
generates $\{(^n)^n\}$ &mdash; "MiniDyck" &mdash; which is not regular.

The reason is that the nonterminal position **alternates**: rightmost in `(A`,
leftmost in `S)`. That alternation is exactly what lets the grammar remember how many
brackets are still open &mdash; unbounded memory, which a DFA cannot have.

## 2. Definitions

### The CFG toolkit

A grammar is a dict; `language`, `nparses`, `parse_trees` and `leftmost` do the work.

In [ ]:
# --- a tiny CFG toolkit -------------------------------------------------
# A grammar is a dict with keys N (nonterminals), Sigma (terminals),
# S (start symbol) and P (productions: nonterminal -> list of RHS tuples).
# A right-hand side is a tuple of one-character symbols; () is epsilon.
# By convention UPPERCASE single letters are nonterminals.

def mkg(rules, start='S'):
    N = set(rules)
    P = {A: [tuple(r) for r in rhs] for A, rhs in rules.items()}
    Sigma = {c for rhs in P.values() for r in rhs for c in r if c not in N}
    return dict(N=N, Sigma=Sigma, S=start, P=P)

def show(G):
    print("N     =", sorted(G['N']))
    print("Sigma =", sorted(G['Sigma']))
    print("S     =", G['S'])
    for A in sorted(G['P']):
        alts = ' | '.join((''.join(r) if r else "''") for r in G['P'][A])
        print("   %s -> %s" % (A, alts))

def derivable(G, maxlen):
    # least fixed point: for each nonterminal, every terminal string of
    # length <= maxlen it derives.  Far cheaper than searching sentential
    # forms, and it terminates because the sets only grow and are bounded.
    T = {A: set() for A in G['N']}
    def spans(r):
        acc = {''}
        for x in r:
            src = T[x] if x in T else {x}
            acc = {a + b for a in acc for b in src if len(a) + len(b) <= maxlen}
            if not acc: break
        return acc
    changed = True
    while changed:
        changed = False
        for A in G['P']:
            for r in G['P'][A]:
                for w in spans(r):
                    if w not in T[A]:
                        T[A].add(w); changed = True
    return T

def language(G, maxlen):
    return sorted(derivable(G, maxlen)[G['S']], key=lambda s: (len(s), s))

def _spans(G, w, cap=None):
    # Bottom-up, shortest span first, so a span never depends on a LONGER
    # one.  Within a span we iterate |N|+1 times, which is enough to close
    # unit rules (A -> B) and epsilon rules.  Doing it top-down with a
    # "cycle guard" silently poisons the memo table, so we do not.
    n, N, P = len(w), G['N'], G['P']
    tab = {}                       # (A, i, j) -> count, or list of trees
    def get(sym, i, j):
        if sym not in N:
            if j == i + 1 and w[i] == sym:
                return 1 if cap is None else [sym]
            return 0 if cap is None else []
        return tab.get((sym, i, j), 0 if cap is None else [])
    def seqv(r, i, j):
        if not r:
            if i != j: return 0 if cap is None else []
            return 1 if cap is None else [()]
        acc = 0 if cap is None else []
        for k in range(i, j + 1):
            a = get(r[0], i, k)
            if not a: continue
            b = seqv(r[1:], k, j)
            if not b: continue
            if cap is None:
                acc += a * b
            else:
                for h in a:
                    for t in b:
                        acc.append((h,) + tuple(t))
                        if len(acc) >= cap: return acc
        return acc
    for length in range(0, n + 1):
        for i in range(0, n - length + 1):
            j = i + length
            for _ in range(len(N) + 1):
                grew = False
                for A in P:
                    v = []
                    for r in P[A]:
                        x = seqv(r, i, j)
                        if cap is None:
                            v.append(x)
                        else:
                            v += [(A,) + tuple(t) for t in x]
                            if len(v) >= cap: v = v[:cap]; break
                    v = sum(v) if cap is None else v
                    old = tab.get((A, i, j), 0 if cap is None else [])
                    if (v != old) if cap is None else (len(v) != len(old)):
                        tab[(A, i, j)] = v; grew = True
                if not grew: break
    return get(G['S'], 0, n)

def nparses(G, w):
    return _spans(G, w, cap=None)

def parse_trees(G, w, cap=8):
    return _spans(G, w, cap=cap)

def yield_of(t):
    return t if isinstance(t, str) else ''.join(yield_of(c) for c in t[1:])

def show_tree(t, ind=0):
    if isinstance(t, str):
        print("%s'%s'" % ('  ' * ind, t)); return
    print("%s%s" % ('  ' * ind, t[0]))
    for c in t[1:]: show_tree(c, ind + 1)

def leftmost(G, w):
    # the leftmost derivation read off one parse tree
    ts = parse_trees(G, w, cap=1)
    if not ts: return None
    steps, form = [], [G['S']]
    def expand(t, pos):
        # t is the subtree rooted at the nonterminal currently at `pos`
        if isinstance(t, str): return pos + 1
        kids = [c if isinstance(c, str) else c[0] for c in t[1:]]
        form[pos:pos+1] = kids
        steps.append(''.join(form) or "''")
        p = pos
        for c in t[1:]:
            p = expand(c, p)
        return p
    steps.append(G['S'])
    expand(ts[0], 0)
    return steps

### The mixed-linear grammar

In [ ]:
Mini = mkg({'S': ["", "(A"], 'A': ["S)"]})
show(Mini)

def linear(G):
    return all(sum(1 for x in r if x in G['N']) <= 1
               for v in G['P'].values() for r in v)
def right_linear(G):
    for v in G['P'].values():
        for r in v:
            nts = [i for i, x in enumerate(r) if x in G['N']]
            if len(nts) > 1 or (nts and nts[0] != len(r) - 1): return False
    return True

### And a right-linear grammar for comparison

In [ ]:
Reg = mkg({'S': ["", "(S"]})

## 3. Tests

The grammar **is** linear, but **not** right-linear and not left-linear.

In [ ]:
print("Mini linear?        ", linear(Mini))
print("Mini right-linear?  ", right_linear(Mini))
assert linear(Mini) and not right_linear(Mini)

It generates MiniDyck: $(^n)^n$.

In [ ]:
L = language(Mini, 8)
print("L(Mini) :", L)
assert all(w == '(' * (len(w)//2) + ')' * (len(w)//2) for w in L)
assert set(L) == {'(' * n + ')' * n for n in range(5)}

Which is **not regular** &mdash; the Pumping Lemma of Chapter 4 settles it.

In [ ]:
def in_mini(s):
    n = len(s) - len(s.lstrip('('))
    return s == '(' * n + ')' * (len(s) - n) and n == len(s) - n

for N in range(2, 7):
    w = '(' * N + ')' * N
    splits = [(w[:i], w[i:j], w[j:]) for i in range(N+1)
              for j in range(i+1, min(N, len(w)) + 1)]
    broken = [1 for x, y, z in splits if any(not in_mini(x + y*k + z) for k in range(3))]
    print("N=%d : %2d splits, %2d broken" % (N, len(splits), len(broken)))
    assert len(broken) == len(splits)
print("\nevery split of (^N )^N pumps out of the language -> not regular")

The right-linear cousin **is** regular, and its DFA is tiny.

In [ ]:
D = md2mc('''DFA
IF : ( -> IF
''')
from itertools import product
assert set(language(Reg, 5)) == {'(' * n for n in range(6)}
print("L(Reg) :", language(Reg, 5))
print("one-state DFA suffices :", all(accepts_dfa(D, '(' * n) for n in range(8)))

**Where the memory comes from:** the nonterminal alternates sides.

In [ ]:
d = leftmost(Mini, '((()))')
for f in d: print("   ", f)
print("\nS -> (A puts the nonterminal on the RIGHT;")
print("A -> S) puts it back on the LEFT.")
print("Each round trip adds one '(' before and one ')' after -- matched growth,")
print("which is the onion idiom, and which no finite state count can track.")

## 4. Exercises


1. Is every **linear** language context-free? Is every CFL linear?
2. Turn `Mini` into a right-linear grammar. What language do you get instead?
3. Where exactly does the "nonterminal is the state" argument of Concept 13 break?

In [ ]:
# Your work for the exercises above.